In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


"Read Bronze Data- Create Data Frame"

In [ ]:
# Load messy raw data into DataFrames from CSVs
df_orders_raw = spark.read.parquet("abfss://e8b72d8d-5a0f-40b4-bcfc-e5ff1552d786@onelake.dfs.fabric.microsoft.com/ac40a136-0e9a-48e0-ab5d-652add7f1fed/Files/bronze/orders_data")
df_returns_raw = spark.read.parquet("abfss://e8b72d8d-5a0f-40b4-bcfc-e5ff1552d786@onelake.dfs.fabric.microsoft.com/ac40a136-0e9a-48e0-ab5d-652add7f1fed/Files/bronze/returns_data")
df_inventory_raw = spark.read.parquet("abfss://e8b72d8d-5a0f-40b4-bcfc-e5ff1552d786@onelake.dfs.fabric.microsoft.com/ac40a136-0e9a-48e0-ab5d-652add7f1fed/Files/bronze/inventory_data")


Handel First row of Returns Data

In [ ]:
#Extract First Row
first_row=df_returns_raw.first()
columns=[str(item).strip() for item in first_row]

#Remove the first row (header row now part of data)
# 1. Convert DataFrame to RDD to use zipWithIndex
# 2. Filter out the first row (index 0)
# 3. Extract just the Row object from the tuple, ignoring the index
rdd_filtered = df_returns_raw.rdd.zipWithIndex() \
    .filter(lambda x: x[1] > 0) \
    .map(lambda x: x[0])

# 4. Reconstruct the DataFrame at the end using your column list
df_returns_raw = spark.createDataFrame(rdd_filtered, schema=columns)

# show cleaned data
display(df_returns_raw)

In [ ]:
display(df_returns_raw)

In [ ]:
%%html
Create Bronze Delta Tables

In [ ]:
# Save raw data to Bronze tables without any transformations
df_orders_raw.write.mode("overwrite").format("delta").saveAsTable("bronze_orders")
df_returns_raw.write.mode("overwrite").format("delta").saveAsTable("bronze_returns")
df_inventory_raw.write.mode("overwrite").format("delta").saveAsTable("bronze_inventory")
